# GAOR classification and adversarial evaluation

This revision covers classification, GA feature selection and ART evaluation. GA selects feature subsets using training data. The ANN uses a separate validation partition.

Outputs are cleared for a fresh run on the prepared dataset. This revision has not been run end to end. Original notebooks and saved outputs are preserved at commit `eec1f50cac9ab65562c70fe1bd1862e9c5b44493`; see README and REVIEW_NOTES.md for version details.


In [ ]:
from pathlib import Path
import os
DATA_PATH = Path(os.environ.get('GAOR_DATA_PATH', 'data/cicddos2019_dataset.csv'))
if not DATA_PATH.is_file():
    raise FileNotFoundError(f'Prepared CSV not found: {DATA_PATH}. See README for schema and provenance requirements.')

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
df = pd.read_csv(DATA_PATH)

In [ ]:
df

In [ ]:
# Print the categorical columns in the dataset
cat_df = df.select_dtypes(include=['object'])
print(cat_df.columns)

In [ ]:
df = df.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

In [ ]:
labels = df['Class'].astype(str).str.strip().str.lower()
if set(labels.unique()) != {'benign', 'attack'}:
    raise ValueError('Expected Class values Benign and Attack. Document any source-label mapping before running.')
df = df.drop(columns=['Unnamed: 0', 'Label'], errors='ignore')
df['Class'] = labels.map({'benign': 0, 'attack': 1})
if len(df.select_dtypes(include='object').columns):
    raise ValueError('Features must be numeric. Document preprocessing rather than silently coercing columns.')

In [ ]:
df

In [ ]:
X = df.drop(['Class'],axis=1)
y = df['Class']

In [ ]:
from sklearn.preprocessing import MinMaxScaler
# Reserve test data first, then a separate validation set for ANN early stopping.
X_dev_raw, X_test_raw, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42)
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_dev_raw, y_dev, test_size=0.20, stratify=y_dev, random_state=42)
scaler = MinMaxScaler(clip=True)
X_train = pd.DataFrame(scaler.fit_transform(X_train_raw), columns=X.columns, index=X_train_raw.index)
X_val = pd.DataFrame(scaler.transform(X_val_raw), columns=X.columns, index=X_val_raw.index)
X_test = pd.DataFrame(scaler.transform(X_test_raw), columns=X.columns, index=X_test_raw.index)
# Clipping unseen extremes to the training range is explicit; report it with results.

In [ ]:
X_train.head()

In [ ]:
print('Train / validation / test rows:', len(X_train), len(X_val), len(X_test))

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

tf.keras.utils.set_random_seed(42)
# Define the model architecture
ann_classifier = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),  # First hidden layer
    Dense(32, activation='relu'),  # Second hidden layer
    Dense(1, activation='sigmoid')  # Output layer for binary classification
])

# Compile the model
ann_classifier.compile(optimizer=Adam(learning_rate=0.001),
                       loss='binary_crossentropy',  # For binary classification, change if needed
                       metrics=['accuracy'])

# Set up early stopping to prevent overfitting
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Train the model with early stopping
history = ann_classifier.fit(X_train, y_train,
                             epochs=100,  # Set a high epoch count to let early stopping handle it
                             batch_size=32,
                             validation_data=(X_val, y_val),
                             callbacks=[early_stopping],
                             verbose=1)

In [ ]:
# Make predictions on the testing data
y_pred_ann = ann_classifier.predict(X_test)
y_pred_ann = (y_pred_ann > 0.5).astype(int)
# Evaluate the model on the test set
loss, accuracy = ann_classifier.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

In [ ]:
# Initialize the Random Forest classifier
rf_classifier = RandomForestClassifier(random_state=42)



# Train the Random Forest classifier on the entire training set
rf_classifier.fit(X_train, y_train)

# Make predictions on the testing data
y_pred = rf_classifier.predict(X_test)

In [ ]:
# Calculate accuracy on the testing data
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

# Print classification report
print("Classification Report:")
print(classification_report(y_test, y_pred, digits=4))

In [ ]:
import xgboost as xgb
# Model training
# Initialize the XGBoost classifier
xg_classifier = xgb.XGBClassifier(objective='binary:logistic', random_state=42)


# Train the Random Forest classifier on the entire training set
xg_classifier.fit(X_train, y_train)

# Make predictions on the testing data
y_pred_xg = xg_classifier.predict(X_test)

In [ ]:
# Calculate accuracy on the testing data
xg_accuracy = accuracy_score(y_test, y_pred_xg)
print("Accuracy:", xg_accuracy)

# Print classification report
print("Classification Report:")
print(classification_report(y_test, y_pred_xg, digits=4))

In [ ]:
# Install dependencies once using requirements.txt before running.

In [ ]:
from genetic_selection import GeneticSelectionCV

In [ ]:
X_train.isna().sum()

In [ ]:
np.random.seed(42)
# Initialize the GeneticSelectionCV object with adjusted parameters
selector = GeneticSelectionCV(estimator=rf_classifier,
                              cv=5,
                              scoring="accuracy",
                              max_features=X_train.shape[1],
                              n_population=10,
                              crossover_proba=0.05,
                              mutation_proba=0.01,
                              n_generations=10,
                              verbose=0,  # Disable verbose output
                              caching=False,  # Disable caching
                              n_jobs=-1)  # Utilize parallel execution if available

# Fit the GeneticSelectionCV object to the training data
selector = selector.fit(X_train, y_train)

# Select features based on the fitted GeneticSelectionCV object
selected_features = X_train.columns[selector.support_]

In [ ]:
# Print the list of selected features
print("List of selected features:", selected_features)
# Print the total number of selected features
print("Total number of selected features:", sum(selector.support_))

In [ ]:
# Print the list of feature names
print("List of original features:", X.columns)
# Print the total number of features
print("Total number of features:", X.shape[1])

In [ ]:
# Transform the training and testing data using the selected features
X_train_selected = selector.transform(X_train)
X_test_selected = selector.transform(X_test)
X_val_selected = selector.transform(X_val)
ga_selected_features = list(selected_features)

In [ ]:
selected_rf_classifier = RandomForestClassifier(random_state=42)


# Train the model on the selected features
selected_rf_classifier.fit(X_train_selected, y_train)

# Predictions on the testing data
selected_y_pred = selected_rf_classifier.predict(X_test_selected)

In [ ]:
# Calculate accuracy on the testing data
selected_accuracy = accuracy_score(y_test, selected_y_pred)
print("Accuracy:", selected_accuracy)

# Print classification report
print("Classification Report:")
print(classification_report(y_test, selected_y_pred, digits=4))

In [ ]:
selected_xg_classifier = xgb.XGBClassifier(objective='binary:logistic', random_state=42)


# Train the model on the selected features
selected_xg_classifier.fit(X_train_selected, y_train)

# Predictions on the testing data
selected_y_pred_xg = selected_xg_classifier.predict(X_test_selected)

In [ ]:
# Calculate accuracy on the testing data
selected_xg_accuracy = accuracy_score(y_test, selected_y_pred_xg)
print("Accuracy:", selected_xg_accuracy)

# Print classification report
print("Classification Report:")
print(classification_report(y_test, selected_y_pred_xg, digits=4))

In [ ]:
# Define the model architecture
selected_ann_classifier = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_selected.shape[1],)),  # First hidden layer
    Dense(32, activation='relu'),  # Second hidden layer
    Dense(1, activation='sigmoid')  # Output layer for binary classification
])

# Compile the model
selected_ann_classifier.compile(optimizer=Adam(learning_rate=0.001),
                       loss='binary_crossentropy',  # For binary classification, change if needed
                       metrics=['accuracy'])

# Train the model with early stopping
selected_history = selected_ann_classifier.fit(X_train_selected, y_train,
                             epochs=20,  # Set a high epoch count to let early stopping handle it
                             batch_size=32,
                             validation_data=(X_val_selected, y_val),
                             callbacks=[early_stopping],
                             verbose=1)

In [ ]:
# Make predictions on the testing data
selected_y_pred_ann = selected_ann_classifier.predict(X_test_selected)
selected_y_pred_ann = (selected_y_pred_ann > 0.5).astype(int)
# Evaluate the model on the test set
selected_loss, selected_accuracy = selected_ann_classifier.evaluate(X_test_selected, y_test, verbose=0)
print(f"Selected Test Loss: {selected_loss:.4f}")
print(f"Selected Test Accuracy: {selected_accuracy:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Plotting confusion matrix for Random Forest without Genetic Algorithm
cm_rf = confusion_matrix(y_test, y_pred)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - Random Forest')
plt.show()

# Plotting confusion matrix for XGBoost without Genetic Algorithm
cm_xg = confusion_matrix(y_test, y_pred_xg)
sns.heatmap(cm_xg, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - XGBoost')
plt.show()

# Plotting confusion matrix for ANN without Genetic Algorithm
cm_ann = confusion_matrix(y_test, y_pred_ann)
sns.heatmap(cm_ann, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - Artificial Neural Network (ANN)')
plt.show()

# Plotting confusion matrix for Random Forest with Genetic Algorithm
selected_cm_rf = confusion_matrix(y_test, selected_y_pred)
sns.heatmap(selected_cm_rf, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - Random Forest with GA')
plt.show()

# Plotting confusion matrix for XGBoost with Genetic Algorithm
selected_cm_xg = confusion_matrix(y_test, selected_y_pred_xg)
sns.heatmap(selected_cm_xg, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - XGBoost with GA')
plt.show()

# Plotting confusion matrix for ANN with Genetic Algorithm
selected_cm_ann = confusion_matrix(y_test, selected_y_pred_ann)
sns.heatmap(selected_cm_ann, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - Artificial Neural Network (ANN) with GA')
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

# Calculate ROC curve and AUC for Random Forest model without Genetic Algorithm
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_classifier.predict_proba(X_test)[:, 1])
auc_rf = roc_auc_score(y_test, rf_classifier.predict_proba(X_test)[:, 1])

# Calculate ROC curve and AUC for Random Forest model with Genetic Algorithm
fpr_rf_ga, tpr_rf_ga, _ = roc_curve(y_test, selected_rf_classifier.predict_proba(X_test_selected)[:, 1])
auc_rf_ga = roc_auc_score(y_test, selected_rf_classifier.predict_proba(X_test_selected)[:, 1])

# Calculate ROC curve and AUC for XGBoost model without Genetic Algorithm
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, xg_classifier.predict_proba(X_test)[:, 1])
auc_xgb = roc_auc_score(y_test, xg_classifier.predict_proba(X_test)[:, 1])

# Calculate ROC curve and AUC for XGBoost model with Genetic Algorithm
fpr_xgb_ga, tpr_xgb_ga, _ = roc_curve(y_test, selected_xg_classifier.predict_proba(X_test_selected)[:, 1])
auc_xgb_ga = roc_auc_score(y_test, selected_xg_classifier.predict_proba(X_test_selected)[:, 1])

# Calculate ROC curve and AUC for ANN model without Genetic Algorithm
fpr_ann, tpr_ann, _ = roc_curve(y_test, ann_classifier.predict(X_test).ravel())
auc_ann = roc_auc_score(y_test, ann_classifier.predict(X_test).ravel())

# Calculate ROC curve and AUC for ANN model with Genetic Algorithm
fpr_ann_ga, tpr_ann_ga, _ = roc_curve(y_test, selected_ann_classifier.predict(X_test_selected).ravel())
auc_ann_ga = roc_auc_score(y_test, selected_ann_classifier.predict(X_test_selected).ravel())

# Plot ROC curves for all models
plt.figure(figsize=(10, 8))
plt.plot(fpr_rf, tpr_rf, label=f'RF without GA (AUC = {auc_rf:.4f})')
plt.plot(fpr_rf_ga, tpr_rf_ga, label=f'RF with GA (AUC = {auc_rf_ga:.4f})')
plt.plot(fpr_xgb, tpr_xgb, label=f'XGB without GA (AUC = {auc_xgb:.4f})')
plt.plot(fpr_xgb_ga, tpr_xgb_ga, label=f'XGB with GA (AUC = {auc_xgb_ga:.4f})')
plt.plot(fpr_ann, tpr_ann, label=f'ANN without GA (AUC = {auc_ann:.4f})')
plt.plot(fpr_ann_ga, tpr_ann_ga, label=f'ANN with GA (AUC = {auc_ann_ga:.4f})')
plt.plot([0, 1], [0, 1], 'k--')  # Diagonal line for random guessing

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves for Random Forest, XGBoost, and ANN Models')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

In [ ]:
# Plot ROC curves
plt.figure(figsize=(8, 6))

plt.plot(fpr_rf, tpr_rf, label=f'RF without GA (AUC = {auc_rf:.4f})')
plt.plot(fpr_rf_ga, tpr_rf_ga, label=f'RF with GA (AUC = {auc_rf_ga:.4f})')
plt.plot(fpr_xgb, tpr_xgb, label=f'XGB without GA (AUC = {auc_xgb:.4f})')
plt.plot(fpr_xgb_ga, tpr_xgb_ga, label=f'XGB with GA (AUC = {auc_xgb_ga:.4f})')
plt.plot(fpr_ann, tpr_ann, label=f'ANN without GA (AUC = {auc_ann:.4f})')
plt.plot(fpr_ann_ga, tpr_ann_ga, label=f'ANN with GA (AUC = {auc_ann_ga:.4f})')

plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Zoomed ROC Curves for Different Models')
plt.legend(loc='lower right')

# Set the limits of the axes
plt.xlim(0, 0.01)
plt.ylim(0.999, 1)

plt.grid(True)
plt.show()

In [ ]:
# Plot feature importances for Random Forest without Genetic Algorithm
plt.figure(figsize=(14, 10))
plt.bar(range(len(rf_classifier.feature_importances_)), rf_classifier.feature_importances_)
plt.xticks(range(len(rf_classifier.feature_importances_)), X.columns, rotation=90)
plt.xlabel('Feature')
plt.ylabel('Importance')
plt.title('Random Forest Feature Importance (without Genetic Algorithm)')
plt.show()

# Plot feature importances for Random Forest with Genetic Algorithm
plt.figure(figsize=(14, 10))
plt.bar(range(len(selected_rf_classifier.feature_importances_)), selected_rf_classifier.feature_importances_)
plt.xticks(range(len(selected_rf_classifier.feature_importances_)), selected_features, rotation=90)
plt.xlabel('Feature')
plt.ylabel('Importance')
plt.title('Random Forest Feature Importance (with Genetic Algorithm)')
plt.show()

In [ ]:
# Plot feature importances for XGBoost without Genetic Algorithm
plt.figure(figsize=(14, 10))
plt.bar(range(len(xg_classifier.feature_importances_)), xg_classifier.feature_importances_)
plt.xticks(range(len(xg_classifier.feature_importances_)), X.columns, rotation=90)
plt.xlabel('Feature')
plt.ylabel('Importance')
plt.title('XGBoost Feature Importance (without Genetic Algorithm)')
plt.show()

# Plot feature importances for XGBoost with Genetic Algorithm
plt.figure(figsize=(14, 8))
plt.bar(range(len(selected_xg_classifier.feature_importances_)), selected_xg_classifier.feature_importances_)
plt.xticks(range(len(selected_xg_classifier.feature_importances_)), selected_features, rotation=90)
plt.xlabel('Feature')
plt.ylabel('Importance')
plt.title('XGBoost Feature Importance (with Genetic Algorithm)')
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve
import matplotlib.pyplot as plt

# Calculate Precision-Recall curve and AUC for Random Forest model without Genetic Algorithm
precision_rf, recall_rf, _ = precision_recall_curve(y_test, rf_classifier.predict_proba(X_test)[:, 1])
plt.plot(recall_rf, precision_rf, label='RF without GA')

# Calculate Precision-Recall curve and AUC for Random Forest model with Genetic Algorithm
precision_rf_ga, recall_rf_ga, _ = precision_recall_curve(y_test, selected_rf_classifier.predict_proba(X_test_selected)[:, 1])
plt.plot(recall_rf_ga, precision_rf_ga, label='RF with GA')

# Calculate Precision-Recall curve and AUC for XGBoost model without Genetic Algorithm
precision_xgb, recall_xgb, _ = precision_recall_curve(y_test, xg_classifier.predict_proba(X_test)[:, 1])
plt.plot(recall_xgb, precision_xgb, label='XGB without GA')

# Calculate Precision-Recall curve and AUC for XGBoost model with Genetic Algorithm
precision_xgb_ga, recall_xgb_ga, _ = precision_recall_curve(y_test, selected_xg_classifier.predict_proba(X_test_selected)[:, 1])
plt.plot(recall_xgb_ga, precision_xgb_ga, label='XGB with GA')

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.grid(True)

# Set the limits of the axes
plt.xlim(0.999, 1.0)
plt.ylim(0.999, 1.0)

plt.show()

In [ ]:
from joblib import dump

# Save Random Forest model without Genetic Algorithm
dump(rf_classifier, 'rf_classifier.joblib')
print("Random Forest model without Genetic Algorithm has been saved as 'rf_classifier.joblib'.")

# Save Random Forest model with Genetic Algorithm
dump(selected_rf_classifier, 'selected_rf_classifier.joblib')
print("Random Forest model with Genetic Algorithm has been saved as 'selected_rf_classifier.joblib'.")

# Save XGBoost model without Genetic Algorithm
dump(xg_classifier, 'xg_classifier.joblib')
print("XGBoost model without Genetic Algorithm has been saved as 'xg_classifier.joblib'.")

# Save XGBoost model with Genetic Algorithm
dump(selected_xg_classifier, 'selected_xg_classifier.joblib')
print("XGBoost model with Genetic Algorithm has been saved as 'selected_xg_classifier.joblib'.")

# Save ANN model without Genetic Algorithm
dump(ann_classifier, 'ann_classifier.joblib')
print("Artificial Neural Network model without Genetic Algorithm has been saved as 'ann_classifier.joblib'.")

# Save ANN model with Genetic Algorithm
dump(selected_ann_classifier, 'selected_ann_classifier.joblib')
print("Artificial Neural Network model with Genetic Algorithm has been saved as 'selected_ann_classifier.joblib'.")

In [ ]:
# Save selected features
dump(selected_features, 'selected_features.joblib')
print("Selected features have been saved as 'selected_features.joblib'.")

In [ ]:
#Loading the saved models
import joblib
rf_classifier = joblib.load('rf_classifier.joblib')
selected_rf_classifier = joblib.load('selected_rf_classifier.joblib')
xg_classifier = joblib.load('xg_classifier.joblib')
selected_xg_classifier = joblib.load('selected_xg_classifier.joblib')

In [ ]:
X_test_selected = X_test[selected_features]

In [ ]:
y_pred = rf_classifier.predict(X_test)
y_pred_xg = xg_classifier.predict(X_test)
selected_y_pred = selected_rf_classifier.predict(X_test_selected)
selected_y_pred_xg = selected_xg_classifier.predict(X_test_selected)

In [ ]:
# Calculate RF accuracy on the testing data

print("Random Forest without GA")
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

# Print classification report
print("Classification Report:")
print(classification_report(y_test, y_pred, digits=4))

In [ ]:
# Calculate RF accuracy with GA on the testing data

print("Random Forest with GA")
selected_accuracy = accuracy_score(y_test, selected_y_pred)
print("Accuracy:", selected_accuracy)

# Print classification report
print("Classification Report:")
print(classification_report(y_test, selected_y_pred, digits=4))

In [ ]:
# Calculate XG accuracy on the testing data

print("XGboost without GA")
xg_accuracy = accuracy_score(y_test, y_pred_xg)
print("Accuracy:", xg_accuracy)

# Print classification report
print("Classification Report:")
print(classification_report(y_test, y_pred_xg, digits=4))

In [ ]:
# Calculate XG accuracy with GA on the testing data

print("XGboost with GA")
selected_xg_accuracy = accuracy_score(y_test, selected_y_pred_xg)
print("Accuracy:", selected_xg_accuracy)

# Print classification report
print("Classification Report:")
print(classification_report(y_test, selected_y_pred_xg, digits=4))

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

# Calculate ROC curve and AUC for Random Forest model without Genetic Algorithm
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_classifier.predict_proba(X_test)[:, 1])
auc_rf = roc_auc_score(y_test, rf_classifier.predict_proba(X_test)[:, 1])

# Calculate ROC curve and AUC for Random Forest model with Genetic Algorithm
fpr_rf_ga, tpr_rf_ga, _ = roc_curve(y_test, selected_rf_classifier.predict_proba(X_test_selected)[:, 1])
auc_rf_ga = roc_auc_score(y_test, selected_rf_classifier.predict_proba(X_test_selected)[:, 1])

# Calculate ROC curve and AUC for XGBoost model without Genetic Algorithm
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, xg_classifier.predict_proba(X_test)[:, 1])
auc_xgb = roc_auc_score(y_test, xg_classifier.predict_proba(X_test)[:, 1])

# Calculate ROC curve and AUC for XGBoost model with Genetic Algorithm
fpr_xgb_ga, tpr_xgb_ga, _ = roc_curve(y_test, selected_xg_classifier.predict_proba(X_test_selected)[:, 1])
auc_xgb_ga = roc_auc_score(y_test, selected_xg_classifier.predict_proba(X_test_selected)[:, 1])

# Plot ROC curves
plt.figure(figsize=(8, 6))
plt.plot(fpr_rf, tpr_rf, label=f'RF without GA (AUC = {auc_rf:.4f})')
plt.plot(fpr_rf_ga, tpr_rf_ga, label=f'RF with GA (AUC = {auc_rf_ga:.4f})')
plt.plot(fpr_xgb, tpr_xgb, label=f'XGB without GA (AUC = {auc_xgb:.4f})')
plt.plot(fpr_xgb_ga, tpr_xgb_ga, label=f'XGB with GA (AUC = {auc_xgb_ga:.4f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves for Different Models')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

In [ ]:
# Create a figure with 1 row and 2 columns of subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 8))

# Plot feature importances for Random Forest without Genetic Algorithm
axes[0].bar(range(len(rf_classifier.feature_importances_)), rf_classifier.feature_importances_)
axes[0].set_xticks(range(len(rf_classifier.feature_importances_)))
axes[0].set_xticklabels(X.columns, rotation=90)
axes[0].set_xlabel('Feature')
axes[0].set_ylabel('Importance')
axes[0].set_title('Random Forest Feature Importance (without GA)')

# Plot feature importances for Random Forest with Genetic Algorithm
axes[1].bar(range(len(selected_rf_classifier.feature_importances_)), selected_rf_classifier.feature_importances_)
axes[1].set_xticks(range(len(selected_rf_classifier.feature_importances_)))
axes[1].set_xticklabels(selected_features, rotation=90)
axes[1].set_xlabel('Feature')
axes[1].set_ylabel('Importance')
axes[1].set_title('Random Forest Feature Importance (with GA)')

# Adjust layout to prevent overlapping labels
plt.tight_layout()

# Display the combined plot
plt.show()

In [ ]:
# Total feature set (all 77)
total_features = list(X.columns)  # or your full set of features
# Selected feature set after GA (34)
selected_features = list(selected_features)  # Loaded from the .joblib file or your selected set

# Find the common features between the total set and the selected set
common_features = list(set(total_features) & set(selected_features))

# Get the indices of the common features in the total feature set
total_indices = [total_features.index(feature) for feature in common_features]
# Get the indices of the common features in the selected feature set
selected_indices = [selected_features.index(feature) for feature in common_features]

# All features (the 77 features)
total_features = list(X.columns)

# Selected features (the 34 features from Genetic Algorithm)
selected_features_set = set(selected_features)  # Convert to set for faster lookup

# Create aligned feature importance arrays with zero for non-existent features
rf_importances_aligned = np.zeros(len(total_features))
selected_rf_importances_aligned = np.zeros(len(total_features))

# Set feature importances for the full set
for i, feature in enumerate(total_features):
    rf_importances_aligned[i] = rf_classifier.feature_importances_[i]

# Set feature importances for the selected set (where it matches the total feature list)
for feature in selected_features_set:
    if feature in total_features:
        idx = total_features.index(feature)  # Find the correct index
        # Assuming feature importances are in the same order as the features
        selected_rf_importances_aligned[idx] = selected_rf_classifier.feature_importances_[selected_features.index(feature)]

# Create the plot with aligned feature importances
num_features = len(total_features)
indices = np.arange(num_features)
bar_width = 0.35  # Width of the bars

plt.figure(figsize=(16, 8))

# Plot feature importances for the full model (77 features)
plt.bar(indices, rf_importances_aligned, bar_width, label='Without GA', color='b')

# Plot feature importances for the selected model (34 features)
plt.bar(indices + bar_width, selected_rf_importances_aligned, bar_width, label='With GA', color='r')

# Set x-ticks to match the full feature list
plt.xticks(indices + bar_width / 2, total_features, rotation=90)

# Labels and title
plt.xlabel('Feature')
plt.ylabel('Importance')
plt.title('Random Forest Feature Importance Comparison (Without GA vs. With GA)')
plt.legend()

# Ensure clean layout and show the plot
plt.tight_layout()  # Adjust layout to avoid overlap
plt.show()

In [ ]:
# Create a figure with 1 row and 2 columns of subplots
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Plot feature importances for XGBoost without Genetic Algorithm
axes[0].bar(range(len(xg_classifier.feature_importances_)), xg_classifier.feature_importances_)
axes[0].set_xticks(range(len(xg_classifier.feature_importances_)))
axes[0].set_xticklabels(X.columns, rotation=90)
axes[0].set_xlabel('Feature')
axes[0].set_ylabel('Importance')
axes[0].set_title('XGBoost Feature Importance (without GA)')

# Plot feature importances for XGBoost with Genetic Algorithm
axes[1].bar(range(len(selected_xg_classifier.feature_importances_)), selected_xg_classifier.feature_importances_)
axes[1].set_xticks(range(len(selected_xg_classifier.feature_importances_)))
axes[1].set_xticklabels(selected_features, rotation=90)
axes[1].set_xlabel('Feature')
axes[1].set_ylabel('Importance')
axes[1].set_title('XGBoost Feature Importance (with GA)')

# Adjust layout to prevent overlapping labels
plt.tight_layout()

# Display the combined plot
plt.show()

In [ ]:
# All features (the 77 features)
total_features = list(X.columns)

# Selected features (the 34 features from Genetic Algorithm)
selected_features_set = set(selected_features)

# Create aligned feature importance arrays with zeros for non-existent features
xg_importances_aligned = np.zeros(len(total_features))
selected_xg_importances_aligned = np.zeros(len(total_features))

# Set feature importances for the full XGBoost model
for i, feature in enumerate(total_features):
    xg_importances_aligned[i] = xg_classifier.feature_importances_[i]

# Set feature importances for the selected XGBoost model (where it matches the total feature list)
for feature in selected_features_set:
    if feature in total_features:
        idx = total_features.index(feature)  # Find the correct index
        selected_xg_importances_aligned[idx] = selected_xg_classifier.feature_importances_[selected_features.index(feature)]

# Create the plot with aligned feature importances for both XGBoost models
num_features = len(total_features)
indices = np.arange(num_features)
bar_width = 0.35  # Width of the bars

plt.figure(figsize=(16, 8))

# Plot feature importances for the full XGBoost model (77 features)
plt.bar(indices, xg_importances_aligned, bar_width, label='XGBoost without GA', color='b')

# Plot feature importances for the XGBoost model with GA (34 features)
plt.bar(indices + bar_width, selected_xg_importances_aligned, bar_width, label='XGBoost with GA', color='r')

# Set x-ticks to align with the total feature list
plt.xticks(indices + bar_width / 2, total_features, rotation=90)

# Labels and title
plt.xlabel('Feature')
plt.ylabel('Importance')
plt.title('XGBoost Feature Importance Comparison (Without GA vs. With GA)')
plt.legend()

# Ensure a clean layout and show the plot
plt.tight_layout()  # Adjust to prevent overlapping labels
plt.show()

In [ ]:
# Displaying all original features of the dataset
total_features

# Extracting Important Features based on the feature importance

In [ ]:
# Set a threshold to determine what "important" means.
# You can set it to a specific percentage or top N features based on importance.
importance_threshold = 0.005  # Features with importance >= 0.5%

# Get the feature importances from Random Forest without GA
rf_importances = rf_classifier.feature_importances_
# Extract the most important features
rf_important_features = [X.columns[i] for i in range(len(rf_importances)) if rf_importances[i] >= importance_threshold]

# Get the feature importances from Random Forest with GA
rf_ga_importances = selected_rf_classifier.feature_importances_
# Extract the most important features from RF with GA
rf_ga_important_features = [selected_features[i] for i in range(len(rf_ga_importances)) if rf_ga_importances[i] >= importance_threshold]

# Get the feature importances from XGBoost without GA
xg_importances = xg_classifier.feature_importances_
# Extract the most important features from XGBoost without GA
xg_important_features = [X.columns[i] for i in range(len(xg_importances)) if xg_importances[i] >= importance_threshold]

# Get the feature importances from XGBoost with GA
xg_ga_importances = selected_xg_classifier.feature_importances_
# Extract the most important features from XGBoost with GA
xg_ga_important_features = [selected_features[i] for i in range(len(xg_ga_importances)) if xg_ga_importances[i] >= importance_threshold]

# Display the extracted important features
print("Important features from Random Forest (without GA):", rf_important_features)
print("Important features from Random Forest (with GA):", rf_ga_important_features)
print("Important features from XGBoost (without GA):", xg_important_features)
print("Important features from XGBoost (with GA):", xg_ga_important_features)

# Printing the resulting test data with the important features

In [ ]:
# Printing the resulting test data with the important features from Random Forest (without GA)
X_test_rf_important = X_test[rf_important_features]

# Printing the resulting test data with the important features from Random Forest (with GA)
X_test_rf_important_ga = X_test[rf_ga_important_features]

# Printing the resulting test data with the important features from XGBoost (without GA)
X_test_xg_important = X_test[xg_important_features]

# Printing the resulting test data with the important features from XGBoost (with GA)
X_test_xg_important_ga = X_test[xg_ga_important_features]

In [ ]:
print("Shape of test data with important features from Random Forest (without GA):", X_test_rf_important.shape)
print("Shape of test data with important features from Random Forest (with GA):", X_test_rf_important_ga.shape)
print("Shape of test data with important features from XGBoost (without GA):", X_test_xg_important.shape)
print("Shape of test data with important features from XGBoost (with GA):", X_test_xg_important_ga.shape)

# Making the resulting test data the same size as the size of the original test data without GA (77) and with GA (34) to avoid errors.
## This is done by making the unimportant features zero (i.e those features in the original test data that are not in the resulting test data)

In [ ]:
# Step 1: Create a DataFrame of zeros with the same columns as X_test
X_test_result_rf = pd.DataFrame(0, index=X_test.index, columns=X_test.columns)
X_test_result_rf_ga = pd.DataFrame(0, index=X_test_selected.index, columns=X_test_selected.columns)
X_test_result_xg = pd.DataFrame(0, index=X_test.index, columns=X_test.columns)
X_test_result_xg_ga = pd.DataFrame(0, index=X_test_selected.index, columns=X_test_selected.columns)

In [ ]:
# Step 2: Fill in known features
# This updates the template with the actual values from the reduced set
X_test_result_rf[X_test_rf_important.columns] = X_test_rf_important
X_test_result_rf_ga[X_test_rf_important_ga.columns] = X_test_rf_important_ga
X_test_result_xg[X_test_xg_important.columns] = X_test_xg_important
X_test_result_xg_ga[X_test_xg_important_ga.columns] = X_test_xg_important_ga

In [ ]:
# Step 3: Check the result
X_test_result_rf.head()

In [ ]:
X_test_result_rf_ga.head()

# Making prediction using the resulting test data

In [ ]:
# Prediction
y_pred_important_rf = rf_classifier.predict(X_test_result_rf)
y_pred_important_rf_ga = selected_rf_classifier.predict(X_test_result_rf_ga)
y_pred_important_xg = rf_classifier.predict(X_test_result_xg)
y_pred_important_xg_ga = selected_rf_classifier.predict(X_test_result_xg_ga)

In [ ]:
print("Random Forest Important without GA")

# Calculate accuracy on the testing data
accuracy_important_rf = accuracy_score(y_test, y_pred_important_rf)
print("Accuracy:", accuracy_important_rf)

# Print classification report
print("Classification Report:")
print(classification_report(y_test, y_pred_important_rf, digits=4))

In [ ]:
print("Random Forest Important with GA")

# Calculate accuracy on the testing data
accuracy_important_rf_ga = accuracy_score(y_test, y_pred_important_rf_ga)
print("Accuracy:", accuracy_important_rf_ga)

# Print classification report
print("Classification Report:")
print(classification_report(y_test, y_pred_important_rf_ga, digits=4))

In [ ]:
print("XGboost Important without GA")

# Calculate accuracy on the testing data
accuracy_important_xg = accuracy_score(y_test, y_pred_important_xg)
print("Accuracy:", accuracy_important_xg)

# Print classification report
print("Classification Report:")
print(classification_report(y_test, y_pred_important_xg, digits=4))

In [ ]:
print("XGboost Important with GA")

# Calculate accuracy on the testing data
accuracy_important_xg_ga = accuracy_score(y_test, y_pred_important_xg_ga)
print("Accuracy:", accuracy_important_xg_ga)

# Print classification report
print("Classification Report:")
print(classification_report(y_test, y_pred_important_xg_ga, digits=4))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
# Plotting confusion matrix for Random Forest without Genetic Algorithm
cm_rf_important = confusion_matrix(y_test, y_pred_important_rf)
sns.heatmap(cm_rf_important, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - Random Forest - Important (Without GA)')
plt.show()

In [ ]:
# Plotting confusion matrix for Random Forest with Genetic Algorithm
cm_rf_important = confusion_matrix(y_test, y_pred_important_rf_ga)
sns.heatmap(cm_rf_important, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - Random Forest - Important (With GA)')
plt.show()

In [ ]:
# Plotting confusion matrix for Random Forest with Genetic Algorithm
cm_rf_important = confusion_matrix(y_test, y_pred_important_xg)
sns.heatmap(cm_rf_important, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - XGboost - Important (Without GA)')
plt.show()

In [ ]:
# Plotting confusion matrix for Random Forest with Genetic Algorithm
cm_rf_important = confusion_matrix(y_test, y_pred_important_xg_ga)
sns.heatmap(cm_rf_important, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - XGboost - Important (With GA)')
plt.show()

# Tweaking the important features to see its effect on the performance (Perturbation)
## Adding the half of the maximum and minimum range to each values of the data

In [ ]:
# Function to add perturbation to the dataset
def add_perturbation(dataset):
    perturbed_dataset = dataset.copy()
    for col_name in dataset.columns:
        col_range = dataset[col_name].max() - dataset[col_name].min()
        perturbation = col_range / 2
        perturbed_dataset[col_name] += perturbation
    return perturbed_dataset

# Perform perturbation on each dataset
X_test_result_rf_perturbed = add_perturbation(X_test_result_rf)
X_test_result_rf_ga_perturbed = add_perturbation(X_test_result_rf_ga)
X_test_result_xg_perturbed = add_perturbation(X_test_result_xg)
X_test_result_xg_ga_perturbed = add_perturbation(X_test_result_xg_ga)

In [ ]:
X_test_result_rf_perturbed.head()

In [ ]:
X_test_result_rf_ga_perturbed.head()

In [ ]:
# Prediction
y_pred_perturbed_rf = rf_classifier.predict(X_test_result_rf_perturbed)
y_pred_perturbed_rf_ga = selected_rf_classifier.predict(X_test_result_rf_ga_perturbed)
y_pred_perturbed_xg = rf_classifier.predict(X_test_result_xg_perturbed)
y_pred_perturbed_xg_ga = selected_rf_classifier.predict(X_test_result_xg_ga_perturbed)

In [ ]:
print("Perturbed Random Forest without GA")

# Calculate accuracy on the testing data
accuracy_perturbed_rf = accuracy_score(y_test, y_pred_perturbed_rf)
print("Accuracy:", accuracy_perturbed_rf)

# Print classification report
print("Classification Report:")
print(classification_report(y_test, y_pred_perturbed_rf, digits=4))

In [ ]:
# Plotting confusion matrix for Perturbed Random Forest without Genetic Algorithm
cm_rf_perturbed = confusion_matrix(y_test, y_pred_perturbed_rf)
sns.heatmap(cm_rf_perturbed, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - Random Forest without GA - Perturbed')
plt.show()

In [ ]:
print("Perturbed Random Forest with GA")

# Calculate accuracy on the testing data
accuracy_perturbed_rf_ga = accuracy_score(y_test, y_pred_perturbed_rf_ga)
print("Accuracy:", accuracy_perturbed_rf_ga)

# Print classification report
print("Classification Report:")
print(classification_report(y_test, y_pred_perturbed_rf_ga, digits=4))

In [ ]:
# Plotting confusion matrix for Perturbed Random Forest with Genetic Algorithm
cm_rf_perturbed_ga = confusion_matrix(y_test, y_pred_perturbed_rf_ga)
sns.heatmap(cm_rf_perturbed_ga, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - Random Forest with GA - Perturbed')
plt.show()

In [ ]:
print("Perturbed XGboost without GA")

# Calculate accuracy on the testing data
accuracy_perturbed_xg = accuracy_score(y_test, y_pred_perturbed_xg)
print("Accuracy:", accuracy_perturbed_xg)

# Print classification report
print("Classification Report:")
print(classification_report(y_test, y_pred_perturbed_xg, digits=4))

In [ ]:
# Plotting confusion matrix for Perturbed XGboost without Genetic Algorithm
cm_xg_perturbed = confusion_matrix(y_test, y_pred_perturbed_xg)
sns.heatmap(cm_xg_perturbed, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - XGboost without GA - Perturbed')
plt.show()

In [ ]:
print("Perturbed XGboost with GA")

# Calculate accuracy on the testing data
accuracy_perturbed_xg_ga = accuracy_score(y_test, y_pred_perturbed_xg_ga)
print("Accuracy:", accuracy_perturbed_xg_ga)

# Print classification report
print("Classification Report:")
print(classification_report(y_test, y_pred_perturbed_xg_ga, digits=4))

In [ ]:
# Plotting confusion matrix for Perturbed XGboost with Genetic Algorithm
cm_xg_perturbed_ga = confusion_matrix(y_test, y_pred_perturbed_xg_ga)
sns.heatmap(cm_xg_perturbed_ga, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - XGboost with GA - Perturbed')
plt.show()

In [ ]:
# Define scenario labels
scenarios = [
    "RF without GA", "RF with GA",
    "XGBoost without GA", "XGBoost with GA",
    "RF Important without GA", "RF Important with GA",
    "XGBoost Important without GA", "XGBoost Important with GA",
    "RF Perturbed without GA", "RF Perturbed with GA",
    "XGBoost Perturbed without GA", "XGBoost Perturbed with GA"
]

# Define accuracy values for each scenario
accuracies = [
    accuracy, selected_accuracy,
    xg_accuracy, selected_xg_accuracy,
    accuracy_important_rf, accuracy_important_rf_ga,
    accuracy_important_xg, accuracy_important_xg_ga,
    accuracy_perturbed_rf, accuracy_perturbed_rf_ga,
    accuracy_perturbed_xg, accuracy_perturbed_xg_ga
]

# Plotting the bar plot
plt.figure(figsize=(12, 6))
plt.bar(scenarios, accuracies, color='skyblue')
plt.xlabel('Scenarios')
plt.ylabel('Accuracy')
plt.title('Accuracy Comparison Across Scenarios')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Save each dataset as a CSV file
X_test_rf_important.to_csv('X_test_rf_important.csv', index=False)
X_test_rf_important_ga.to_csv('X_test_rf_important_ga.csv', index=False)
X_test_xg_important.to_csv('X_test_xg_important.csv', index=False)
X_test_xg_important_ga.to_csv('X_test_xg_important_ga.csv', index=False)

In [ ]:
X_test_rf_important

In [ ]:
# Exported CSV files are saved in the current working directory.

In [ ]:
# No Google Colab account or Drive mount is required.

## ART adversarial evaluation

This section evaluates fitted Random Forest and XGBoost models with full and GA-selected features. It is a subsequent evaluation procedure, separate from the paper's numerical attack table.

The default five-row subset is a smoke test, shared across models and including both classes. For quantitative results, increase the sample count, report denominators and repeat seeds. Continuous feature-space perturbations are bounded to [0, 1] without network-traffic validity constraints.


In [ ]:
from art.attacks.evasion import ZooAttack
from art.estimators.classification import BlackBoxClassifier
from art.utils import to_categorical

ATTACK_SAMPLE_COUNT = 5  # Smoke test, not a robustness benchmark.
if not 2 <= ATTACK_SAMPLE_COUNT <= len(y_test):
    raise ValueError('Choose an attack sample count between 2 and the test-set size.')
# Randomly select at least one row per class, then fill the remaining slots.
rng = np.random.default_rng(42)
labels = y_test.to_numpy(dtype=int)
chosen = [int(rng.choice(np.flatnonzero(labels == label))) for label in (0, 1)]
remaining = np.setdiff1d(np.arange(len(labels)), chosen)
chosen += rng.choice(remaining, ATTACK_SAMPLE_COUNT - 2, replace=False).tolist()
attack_indices = np.asarray(chosen)
y_attack = labels[attack_indices]
x_attack_full = X_test.to_numpy(dtype=np.float32)[attack_indices]
x_attack_selected = selector.transform(X_test).astype(np.float32)[attack_indices]
print('Exploratory attack subset:', len(y_attack), 'rows; class counts:', np.bincount(y_attack))

In [ ]:
def evaluate_predictions(probabilities, labels):
    labels = np.asarray(labels)
    if labels.ndim == 2 and labels.shape[1] == 1:
        labels = labels[:, 0]
    if labels.ndim != 1:
        raise ValueError('Reference labels must be class IDs, not a one-hot matrix.')
    probabilities = np.asarray(probabilities)
    if probabilities.shape != (len(labels), 2) or not np.isfinite(probabilities).all():
        raise ValueError('Expected a finite n-by-2 class probability matrix.')
    predicted = probabilities.argmax(axis=1)
    return float(accuracy_score(labels, predicted)), predicted

attack_results = []
for model_name, fitted_model, inputs, feature_names in [
    ('RF full', rf_classifier, x_attack_full, list(X.columns)),
    ('RF GA selected', selected_rf_classifier, x_attack_selected, None),
    ('XGBoost full', xg_classifier, x_attack_full, list(X.columns)),
    ('XGBoost GA selected', selected_xg_classifier, x_attack_selected, None),
]:
    def predict_probabilities(values, model=fitted_model, names=feature_names):
        frame = pd.DataFrame(values, columns=names) if names is not None else values
        return model.predict_proba(frame)
    classifier = BlackBoxClassifier(
        predict_fn=predict_probabilities, input_shape=(inputs.shape[1],),
        nb_classes=2, clip_values=(0.0, 1.0))
    np.random.seed(42)
    clean_accuracy, clean_predictions = evaluate_predictions(classifier.predict(inputs), y_attack)
    attack = ZooAttack(classifier=classifier, confidence=0.0, targeted=False,
        learning_rate=0.1, max_iter=200, binary_search_steps=10,
        initial_const=0.001, abort_early=True, use_resize=False,
        use_importance=False, nb_parallel=min(5, inputs.shape[1]),
        batch_size=1, variable_h=0.01)
    adversarial_inputs = attack.generate(x=inputs.copy(), y=to_categorical(y_attack, nb_classes=2))
    adversarial_accuracy, adversarial_predictions = evaluate_predictions(classifier.predict(adversarial_inputs), y_attack)
    correct = clean_predictions == y_attack
    success = float(np.mean(adversarial_predictions[correct] != y_attack[correct])) if correct.any() else None
    attack_results.append({'model': model_name, 'n': len(y_attack),
        'clean_accuracy': clean_accuracy, 'adversarial_accuracy': adversarial_accuracy,
        'attack_success_on_initially_correct': success,
        'initially_correct_n': int(correct.sum())})
pd.DataFrame(attack_results)